In [1]:
# 02_encode_inpta_dr2_noise_budget.ipynb
import numpy as np, pandas as pd

# Noise budget from Nobleson et al. 2025 (arXiv:2512.20455), Table 5.
# b3/b5 = median ToA uncertainty (us) per band; None = band absent for that pulsar.
# arn/dmn = (log10_A, gamma); fcn = (log10_A, gamma, chi_chromatic_index)
noise = {
 'J0030+0451': dict(b3=3.91, b5=None, arn=None, dmn=None, fcn=None),
 'J0034-0534': dict(b3=2.17, b5=None, arn=None, dmn=None, fcn=None),
 'J0437-4715': dict(b3=1.44, b5=1.25, arn=(-12.36,0.37), dmn=(-13.10,1.17), fcn=(-11.91,5.03,0.60)),
 'J0613-0200': dict(b3=1.27, b5=6.12, arn=(-12.11,1.58), dmn=(-13.43,1.47), fcn=None),
 'J0645+5158': dict(b3=14.82,b5=24.54,arn=None, dmn=None, fcn=None),
 'J0740+6620': dict(b3=2.50, b5=5.35, arn=None, dmn=None, fcn=None),
 'J0751+1807': dict(b3=7.79, b5=10.94,arn=None, dmn=None, fcn=None),
 'J0900-3144': dict(b3=None, b5=13.95,arn=None, dmn=None, fcn=None),
 'J1012+5307': dict(b3=6.62, b5=6.77, arn=(-12.38,2.70), dmn=None, fcn=None),
 'J1022+1001': dict(b3=7.03, b5=17.02,arn=None, dmn=(-13.12,0.40), fcn=(-16.18,0.49,6.68)),
 'J1024-0719': dict(b3=27.70,b5=28.55,arn=None, dmn=None, fcn=None),
 'J1125+7819': dict(b3=1.71, b5=None, arn=None, dmn=None, fcn=None),
 'J1455-3330': dict(b3=24.41,b5=21.20,arn=None, dmn=None, fcn=None),
 'J1600-3053': dict(b3=4.98, b5=4.35, arn=None, dmn=(-13.01,1.45), fcn=None),
 'J1614-2230': dict(b3=25.35,b5=23.65,arn=None, dmn=None, fcn=None),
 'J1640+2224': dict(b3=12.37,b5=12.73,arn=None, dmn=None, fcn=None),
 'J1643-1224': dict(b3=5.08, b5=7.03, arn=(-11.09,3.27), dmn=(-12.07,3.22), fcn=(-10.93,2.75,0.66)),
 'J1713+0747': dict(b3=4.13, b5=4.76, arn=None, dmn=(-13.81,1.44), fcn=(-16.23,4.58,5.31)),
 'J1730-2304': dict(b3=5.39, b5=17.26,arn=None, dmn=(-13.35,1.37), fcn=None),
 'J1744-1134': dict(b3=1.21, b5=2.32, arn=None, dmn=(-13.65,1.08), fcn=None),
 'J1857+0943': dict(b3=3.50, b5=4.31, arn=None, dmn=(-13.47,2.13), fcn=None),
 'J1909-3744': dict(b3=0.72, b5=1.21, arn=(-12.94,0.52), dmn=(-13.54,1.36), fcn=(-14.85,3.01,3.40)),
 'J1939+2134': dict(b3=0.44, b5=3.55, arn=(-12.04,2.21), dmn=(-12.86,2.40), fcn=(-15.01,2.68,5.22)),
 'J1944+0907': dict(b3=7.78, b5=22.59,arn=None, dmn=(-12.86,3.36), fcn=None),
 'J2124-3358': dict(b3=7.94, b5=13.50,arn=None, dmn=(-13.60,0.31), fcn=None),
 'J2145-0750': dict(b3=1.98, b5=6.71, arn=None, dmn=(-13.59,0.56), fcn=(-16.44,4.03,6.91)),  # DMN gamma=0.56 verified from Table 5 (arXiv:2512.20455v2 HTML), 65 harmonics
 'J2302+4442': dict(b3=6.77, b5=None, arn=None, dmn=None, fcn=None),
}

# flatten to columns
recs=[]
for psr,v in noise.items():
    recs.append(dict(
        psr=psr, toa_b3_us=v['b3'], toa_b5_us=v['b5'],
        arn_log10A=v['arn'][0] if v['arn'] else np.nan,
        arn_gamma =v['arn'][1] if v['arn'] else np.nan,
        dmn_log10A=v['dmn'][0] if v['dmn'] else np.nan,
        dmn_gamma =v['dmn'][1] if v['dmn'] else np.nan,
        fcn_log10A=v['fcn'][0] if v['fcn'] else np.nan,
        fcn_gamma =v['fcn'][1] if v['fcn'] else np.nan,
        fcn_chi   =v['fcn'][2] if v['fcn'] else np.nan,
    ))
ndf = pd.DataFrame(recs)

# validation: counts must match paper Table 2
assert ndf.arn_log10A.notna().sum()==6,  "ARN count != 6"
assert ndf.dmn_log10A.notna().sum()==14, "DMN count != 14"
assert ndf.fcn_log10A.notna().sum()==7,  "FCN count != 7"

# merge onto geometry table from step 1a
geo = pd.read_csv("inpta_dr2_geometry_whitenoise.csv")
full = geo.merge(ndf, on="psr", how="left")
assert len(full)==27 and full.toa_b3_us.notna().sum()+full.toa_b5_us.notna().sum()>0
full.to_csv("inpta_dr2_full_input_table.csv", index=False)
print(f"Merged table: {full.shape[0]} pulsars x {full.shape[1]} columns")
print(f"ARN/DMN/FCN counts: {ndf.arn_log10A.notna().sum()}/{ndf.dmn_log10A.notna().sum()}/{ndf.fcn_log10A.notna().sum()} (paper: 6/14/7)")
full[['psr','span_yr','toa_b3_us','toa_b5_us','arn_log10A','dmn_log10A','fcn_log10A','fcn_chi']]

Merged table: 27 pulsars x 20 columns
ARN/DMN/FCN counts: 6/14/7 (paper: 6/14/7)


,psr,span_yr,toa_b3_us,toa_b5_us,arn_log10A,dmn_log10A,fcn_log10A,fcn_chi
0,J1857+0943,7.199873,3.50,4.31,NaN,-13.47,NaN,NaN
1,J1730-2304,7.199664,5.39,17.26,NaN,-13.35,NaN,NaN
2,J1909-3744,7.199657,0.72,1.21,-12.94,-13.54,-14.85,3.40
3,J0751+1807,7.188856,7.79,10.94,NaN,NaN,NaN,NaN
4,J1022+1001,7.188649,7.03,17.02,NaN,-13.12,-16.18,6.68
5,J0613-0200,7.185894,1.27,6.12,-12.11,-13.43,NaN,NaN
6,J1012+5307,7.101463,6.62,6.77,-12.38,NaN,NaN,NaN
7,J0437-4715,6.817833,1.44,1.25,-12.36,-13.10,-11.91,0.60
8,J2302+4442,6.667477,6.77,NaN,NaN,NaN,NaN,NaN
9,J2124-3358,5.903194,7.94,13.50,NaN,-13.60,NaN,NaN
